In [1]:
import numpy as np
from collections import defaultdict

class PCFG:
    """
    Probabilistic Context-Free Grammar implementation with Inside-Outside algorithms.
    """
    
    def __init__(self):
        # Non-terminal rules: A -> B C with probability p
        # Format: binary_rules[A] = [(B, C, prob), ...]
        self.binary_rules = defaultdict(list)
        
        # Lexical rules: A -> word with probability p
        # Format: lexical_rules[A] = [(word, prob), ...]
        self.lexical_rules = defaultdict(list)
        
        # Reverse index for binary rules: given (B, C), find all A -> B C
        self.binary_rules_by_children = defaultdict(list)
        
        # Reverse index for lexical rules: given word, find all A -> word
        self.lexical_rules_by_word = defaultdict(list)
        
        # Set of all non-terminals
        self.nonterminals = set()
        
        # Start symbol
        self.start_symbol = 'S'
    
    def add_binary_rule(self, lhs, rhs1, rhs2, prob):
        """Add a binary rule: lhs -> rhs1 rhs2 with given probability."""
        self.binary_rules[lhs].append((rhs1, rhs2, prob))
        self.binary_rules_by_children[(rhs1, rhs2)].append((lhs, prob))
        self.nonterminals.add(lhs)
        self.nonterminals.add(rhs1)
        self.nonterminals.add(rhs2)
    
    def add_lexical_rule(self, lhs, word, prob):
        """Add a lexical rule: lhs -> word with given probability."""
        self.lexical_rules[lhs].append((word, prob))
        self.lexical_rules_by_word[word].append((lhs, prob))
        self.nonterminals.add(lhs)
    
    def get_rule_prob(self, lhs, rhs1, rhs2):
        """Get probability of rule lhs -> rhs1 rhs2."""
        for r1, r2, prob in self.binary_rules[lhs]:
            if r1 == rhs1 and r2 == rhs2:
                return prob
        return 0.0
    
    def get_lexical_prob(self, lhs, word):
        """Get probability of rule lhs -> word."""
        for w, prob in self.lexical_rules[lhs]:
            if w == word:
                return prob
        return 0.0


def create_grammar():
    """
    Create the PCFG from the problem specification.
    """
    grammar = PCFG()
    
    # Non-terminal (binary) rules
    grammar.add_binary_rule('S', 'NP', 'VP', 1.0)
    grammar.add_binary_rule('PP', 'P', 'NP', 1.0)
    grammar.add_binary_rule('VP', 'V', 'NP', 0.7)
    grammar.add_binary_rule('VP', 'VP', 'PP', 0.3)
    grammar.add_binary_rule('NP', 'NP', 'PP', 0.4)
    
    # Lexical (terminal) rules
    grammar.add_lexical_rule('NP', 'astronomers', 0.1)
    grammar.add_lexical_rule('NP', 'ears', 0.18)
    grammar.add_lexical_rule('NP', 'saw', 0.04)
    grammar.add_lexical_rule('NP', 'stars', 0.18)
    grammar.add_lexical_rule('NP', 'telescopes', 0.1)
    grammar.add_lexical_rule('P', 'with', 1.0)
    grammar.add_lexical_rule('V', 'saw', 1.0)
    
    return grammar


class InsideOutsideAlgorithm:
    """
    Implementation of the Inside-Outside algorithm for PCFGs.
    
    The inside probability α(A, i, j) is the probability that non-terminal A
    generates the substring w[i:j+1].
    
    The outside probability β(A, i, j) is the probability that the start symbol
    generates w[0:i] from some derivation involving A at position (i,j), 
    times the probability of generating w[j+1:n] from the rest of that derivation.
    """
    
    def __init__(self, grammar):
        self.grammar = grammar
    
    def inside_algorithm(self, sentence):
        """
        Compute inside probabilities for all non-terminals and spans.
        
        α(A, i, j) = P(A =>* w_i ... w_j)
        
        This is the probability that non-terminal A derives the substring
        from position i to position j (inclusive).
        
        Args:
            sentence: List of words
            
        Returns:
            inside: Dictionary mapping (A, i, j) to inside probability
        """
        n = len(sentence)
        inside = defaultdict(float)
        
        print("=" * 70)
        print("INSIDE ALGORITHM")
        print("=" * 70)
        print(f"\nSentence: {' '.join(sentence)}")
        print(f"Length: {n}")
        print("\nThe inside probability α(A, i, j) represents:")
        print("P(A =>* w_i w_{i+1} ... w_j)")
        print("i.e., the probability that non-terminal A generates words i through j")
        
        # Base case: spans of length 1 (lexical rules)
        print("\n" + "-" * 70)
        print("STEP 1: Base Case (Lexical Rules) - Spans of Length 1")
        print("-" * 70)
        
        for i in range(n):
            word = sentence[i]
            print(f"\nPosition {i}, word = '{word}':")
            
            for lhs, prob in self.grammar.lexical_rules_by_word[word]:
                inside[(lhs, i, i)] = prob
                print(f"  α({lhs}, {i}, {i}) = P({lhs} -> '{word}') = {prob}")
        
        # Recursive case: spans of length > 1
        print("\n" + "-" * 70)
        print("STEP 2: Recursive Case (Binary Rules) - Spans of Length > 1")
        print("-" * 70)
        print("\nFor each span (i, j) and non-terminal A:")
        print("α(A, i, j) = Σ_{A->BC} Σ_{k=i}^{j-1} P(A->BC) × α(B, i, k) × α(C, k+1, j)")
        
        for span_length in range(2, n + 1):
            print(f"\n{'~' * 60}")
            print(f"Processing spans of length {span_length}")
            print(f"{'~' * 60}")
            
            for i in range(n - span_length + 1):
                j = i + span_length - 1
                print(f"\n  Span ({i}, {j}): '{' '.join(sentence[i:j+1])}'")
                
                # Try all possible split points
                for k in range(i, j):
                    print(f"    Split point k={k}: [{i}:{k}] + [{k+1}:{j}]")
                    
                    # Try all binary rules A -> B C
                    for A in self.grammar.nonterminals:
                        for B, C, rule_prob in self.grammar.binary_rules[A]:
                            left_prob = inside[(B, i, k)]
                            right_prob = inside[(C, k + 1, j)]
                            
                            if left_prob > 0 and right_prob > 0:
                                contribution = rule_prob * left_prob * right_prob
                                inside[(A, i, j)] += contribution
                                
                                print(f"      {A} -> {B} {C}:")
                                print(f"        P({A}->{B} {C}) × α({B},{i},{k}) × α({C},{k+1},{j})")
                                print(f"        = {rule_prob} × {left_prob} × {right_prob}")
                                print(f"        = {contribution}")
                
                # Print final inside probability for this span
                for A in self.grammar.nonterminals:
                    if inside[(A, i, j)] > 0:
                        print(f"    => α({A}, {i}, {j}) = {inside[(A, i, j)]:.10f}")
        
        # Final result
        print("\n" + "=" * 70)
        print("INSIDE ALGORITHM RESULT")
        print("=" * 70)
        
        total_prob = inside[(self.grammar.start_symbol, 0, n - 1)]
        print(f"\nP(sentence) = α({self.grammar.start_symbol}, 0, {n-1}) = {total_prob:.10f}")
        print("\nThis is the total probability of generating the sentence from the grammar.")
        
        return inside
    
    def outside_algorithm(self, sentence, inside):
        """
        Compute outside probabilities for all non-terminals and spans.
        
        β(A, i, j) = P(S =>* w_0...w_{i-1} A w_{j+1}...w_{n-1})
        
        This is the probability that the start symbol derives a string where:
        - w_0 to w_{i-1} appears before the position where A is used
        - w_{j+1} to w_{n-1} appears after the position where A is used
        
        Args:
            sentence: List of words
            inside: Inside probabilities computed by inside_algorithm
            
        Returns:
            outside: Dictionary mapping (A, i, j) to outside probability
        """
        n = len(sentence)
        outside = defaultdict(float)
        
        print("\n" + "=" * 70)
        print("OUTSIDE ALGORITHM")
        print("=" * 70)
        print(f"\nSentence: {' '.join(sentence)}")
        print("\nThe outside probability β(A, i, j) represents:")
        print("P(S =>* w_0...w_{i-1} A w_{j+1}...w_{n-1})")
        print("i.e., the probability of everything 'outside' the span (i,j)")
        print("when A is used to derive that span")
        
        # Base case: β(S, 0, n-1) = 1
        print("\n" + "-" * 70)
        print("STEP 1: Base Case")
        print("-" * 70)
        outside[(self.grammar.start_symbol, 0, n - 1)] = 1.0
        print(f"\nβ({self.grammar.start_symbol}, 0, {n-1}) = 1.0")
        print("(The start symbol spanning the entire sentence has outside probability 1)")
        
        # Recursive case: process spans from longest to shortest
        print("\n" + "-" * 70)
        print("STEP 2: Recursive Case - Process Spans from Longest to Shortest")
        print("-" * 70)
        print("\nFor rules A -> B C, we update:")
        print("  β(B, i, k) += β(A, i, j) × P(A->BC) × α(C, k+1, j)  [B is left child]")
        print("  β(C, k+1, j) += β(A, i, j) × P(A->BC) × α(B, i, k)  [C is right child]")
        
        for span_length in range(n, 0, -1):
            print(f"\n{'~' * 60}")
            print(f"Processing spans of length {span_length}")
            print(f"{'~' * 60}")
            
            for i in range(n - span_length + 1):
                j = i + span_length - 1
                
                for A in self.grammar.nonterminals:
                    if outside[(A, i, j)] == 0:
                        continue
                    
                    print(f"\n  Non-terminal {A} at span ({i}, {j}), β({A},{i},{j}) = {outside[(A, i, j)]:.10f}")
                    
                    # For each rule A -> B C where A spans (i, j)
                    for B, C, rule_prob in self.grammar.binary_rules[A]:
                        print(f"    Rule: {A} -> {B} {C} (prob = {rule_prob})")
                        
                        # Try all split points within this span
                        for k in range(i, j):
                            left_inside = inside[(B, i, k)]
                            right_inside = inside[(C, k + 1, j)]
                            
                            if left_inside > 0 and right_inside > 0:
                                # Update outside probability for left child B
                                contrib_left = outside[(A, i, j)] * rule_prob * right_inside
                                outside[(B, i, k)] += contrib_left
                                print(f"      Split k={k}:")
                                print(f"        β({B},{i},{k}) += β({A},{i},{j}) × P({A}->{B}{C}) × α({C},{k+1},{j})")
                                print(f"                      = {outside[(A, i, j)]:.10f} × {rule_prob} × {right_inside:.10f}")
                                print(f"                      = {contrib_left:.10f}")
                                
                                # Update outside probability for right child C
                                contrib_right = outside[(A, i, j)] * rule_prob * left_inside
                                outside[(C, k + 1, j)] += contrib_right
                                print(f"        β({C},{k+1},{j}) += β({A},{i},{j}) × P({A}->{B}{C}) × α({B},{i},{k})")
                                print(f"                       = {outside[(A, i, j)]:.10f} × {rule_prob} × {left_inside:.10f}")
                                print(f"                       = {contrib_right:.10f}")
        
        print("\n" + "=" * 70)
        print("OUTSIDE PROBABILITIES SUMMARY")
        print("=" * 70)
        
        for span_length in range(1, n + 1):
            for i in range(n - span_length + 1):
                j = i + span_length - 1
                for A in sorted(self.grammar.nonterminals):
                    if outside[(A, i, j)] > 0:
                        print(f"β({A}, {i}, {j}) = {outside[(A, i, j)]:.10f}")
        
        return outside
    
    def compute_expected_counts(self, sentence, inside, outside):
        """
        Compute expected counts of rules using inside-outside probabilities.
        
        These expected counts are used in the M-step of the EM algorithm
        for grammar induction.
        
        E[count(A -> B C)] = Σ_{i,j,k} (β(A,i,j) × P(A->BC) × α(B,i,k) × α(C,k+1,j)) / P(sentence)
        E[count(A -> w)] = Σ_i (β(A,i,i) × P(A->w)) / P(sentence)  where w = sentence[i]
        """
        n = len(sentence)
        sentence_prob = inside[(self.grammar.start_symbol, 0, n - 1)]
        
        print("\n" + "=" * 70)
        print("EXPECTED RULE COUNTS")
        print("=" * 70)
        print(f"\nSentence probability P(S) = {sentence_prob:.10f}")
        print("\nExpected counts are computed as:")
        print("E[A->BC] = Σ (β(A,i,j) × P(A->BC) × α(B,i,k) × α(C,k+1,j)) / P(sentence)")
        print("E[A->w]  = Σ (β(A,i,i) × P(A->w)) / P(sentence)")
        
        if sentence_prob == 0:
            print("\nSentence probability is 0 - cannot compute expected counts")
            return {}, {}
        
        # Expected counts for binary rules
        binary_counts = defaultdict(float)
        print("\n" + "-" * 70)
        print("Expected Counts for Binary Rules")
        print("-" * 70)
        
        for A in self.grammar.nonterminals:
            for B, C, rule_prob in self.grammar.binary_rules[A]:
                count = 0.0
                contributions = []
                
                for span_length in range(2, n + 1):
                    for i in range(n - span_length + 1):
                        j = i + span_length - 1
                        
                        for k in range(i, j):
                            outside_A = outside[(A, i, j)]
                            inside_B = inside[(B, i, k)]
                            inside_C = inside[(C, k + 1, j)]
                            
                            if outside_A > 0 and inside_B > 0 and inside_C > 0:
                                contrib = (outside_A * rule_prob * inside_B * inside_C) / sentence_prob
                                count += contrib
                                contributions.append((i, j, k, contrib))
                
                if count > 0:
                    binary_counts[(A, B, C)] = count
                    print(f"\nE[{A} -> {B} {C}] = {count:.10f}")
                    for i, j, k, contrib in contributions:
                        print(f"  Span ({i},{j}), split {k}: contribution = {contrib:.10f}")
        
        # Expected counts for lexical rules
        lexical_counts = defaultdict(float)
        print("\n" + "-" * 70)
        print("Expected Counts for Lexical Rules")
        print("-" * 70)
        
        for i in range(n):
            word = sentence[i]
            for A, prob in self.grammar.lexical_rules_by_word[word]:
                outside_A = outside[(A, i, i)]
                if outside_A > 0:
                    count = (outside_A * prob) / sentence_prob
                    lexical_counts[(A, word)] += count
                    print(f"\nE[{A} -> '{word}'] at position {i}:")
                    print(f"  = β({A},{i},{i}) × P({A}->'{word}') / P(sentence)")
                    print(f"  = {outside_A:.10f} × {prob} / {sentence_prob:.10f}")
                    print(f"  = {count:.10f}")
        
        print("\n" + "-" * 70)
        print("Summary of Expected Counts")
        print("-" * 70)
        
        print("\nBinary Rules:")
        for (A, B, C), count in sorted(binary_counts.items()):
            print(f"  E[{A} -> {B} {C}] = {count:.10f}")
        
        print("\nLexical Rules:")
        for (A, word), count in sorted(lexical_counts.items()):
            print(f"  E[{A} -> '{word}'] = {count:.10f}")
        
        return binary_counts, lexical_counts
    
    def verify_probabilities(self, sentence, inside, outside):
        """
        Verify that inside and outside probabilities are consistent.
        
        For any non-terminal A and span (i,j):
        P(sentence) = α(A, i, j) × β(A, i, j) summed appropriately
        
        More precisely:
        P(sentence) = Σ_A α(A, i, j) × β(A, i, j) × P(A spans (i,j) in some derivation)
        """
        n = len(sentence)
        sentence_prob = inside[(self.grammar.start_symbol, 0, n - 1)]
        
        print("\n" + "=" * 70)
        print("VERIFICATION: Inside × Outside Products")
        print("=" * 70)
        print(f"\nSentence probability from inside algorithm: {sentence_prob:.10f}")
        print("\nFor each span, α(A,i,j) × β(A,i,j) gives the probability of all")
        print("derivations where A generates the span (i,j).")
        
        for span_length in range(1, n + 1):
            print(f"\n--- Spans of length {span_length} ---")
            for i in range(n - span_length + 1):
                j = i + span_length - 1
                print(f"\nSpan ({i},{j}): '{' '.join(sentence[i:j+1])}'")
                
                for A in sorted(self.grammar.nonterminals):
                    inside_val = inside[(A, i, j)]
                    outside_val = outside[(A, i, j)]
                    
                    if inside_val > 0 or outside_val > 0:
                        product = inside_val * outside_val
                        print(f"  {A}: α={inside_val:.10f}, β={outside_val:.10f}, α×β={product:.10f}")


def print_grammar(grammar):
    """Print the grammar rules in a formatted way."""
    print("=" * 70)
    print("PROBABILISTIC CONTEXT-FREE GRAMMAR")
    print("=" * 70)
    
    print("\nBinary Rules (A -> B C):")
    print("-" * 40)
    for A in sorted(grammar.binary_rules.keys()):
        for B, C, prob in grammar.binary_rules[A]:
            print(f"  {A} -> {B} {C}  [{prob}]")
    
    print("\nLexical Rules (A -> word):")
    print("-" * 40)
    for A in sorted(grammar.lexical_rules.keys()):
        for word, prob in grammar.lexical_rules[A]:
            print(f"  {A} -> '{word}'  [{prob}]")
    
    print()


def visualize_chart(sentence, inside, nonterminals):
    """Create a visual representation of the CKY chart with inside probabilities."""
    n = len(sentence)
    
    print("\n" + "=" * 70)
    print("CKY CHART VISUALIZATION (Inside Probabilities)")
    print("=" * 70)
    
    # Create chart display
    print("\nChart[i][j] shows non-terminals that can derive words i through j")
    print("with their inside probabilities.\n")
    
    for span_length in range(1, n + 1):
        print(f"Span length {span_length}:")
        for i in range(n - span_length + 1):
            j = i + span_length - 1
            entries = []
            for A in sorted(nonterminals):
                if inside[(A, i, j)] > 0:
                    entries.append(f"{A}:{inside[(A, i, j)]:.6f}")
            
            if entries:
                span_text = ' '.join(sentence[i:j+1])
                print(f"  [{i},{j}] '{span_text}': {', '.join(entries)}")
        print()


def main():
    """Main function to demonstrate the Inside-Outside algorithm."""
    
    # Create the grammar
    grammar = create_grammar()
    print_grammar(grammar)
    
    # Test sentence
    sentence = ["astronomers", "saw", "stars", "with", "telescopes"]
    
    print("=" * 70)
    print(f"PARSING SENTENCE: {' '.join(sentence)}")
    print("=" * 70)
    
    # Create the algorithm instance
    algorithm = InsideOutsideAlgorithm(grammar)
    
    # Run inside algorithm
    inside = algorithm.inside_algorithm(sentence)
    
    # Visualize the chart
    visualize_chart(sentence, inside, grammar.nonterminals)
    
    # Run outside algorithm
    outside = algorithm.outside_algorithm(sentence, inside)
    
    # Compute expected counts
    binary_counts, lexical_counts = algorithm.compute_expected_counts(
        sentence, inside, outside
    )
    
    # Verify probabilities
    algorithm.verify_probabilities(sentence, inside, outside)
    
    # Additional example with a shorter sentence
    print("\n" + "#" * 70)
    print("ADDITIONAL EXAMPLE: Shorter Sentence")
    print("#" * 70)
    
    sentence2 = ["astronomers", "saw", "stars"]
    
    print(f"\nParsing: {' '.join(sentence2)}")
    inside2 = algorithm.inside_algorithm(sentence2)
    visualize_chart(sentence2, inside2, grammar.nonterminals)
    outside2 = algorithm.outside_algorithm(sentence2, inside2)
    algorithm.compute_expected_counts(sentence2, inside2, outside2)


if __name__ == "__main__":
    main()

PROBABILISTIC CONTEXT-FREE GRAMMAR

Binary Rules (A -> B C):
----------------------------------------
  NP -> NP PP  [0.4]
  PP -> P NP  [1.0]
  S -> NP VP  [1.0]
  VP -> V NP  [0.7]
  VP -> VP PP  [0.3]

Lexical Rules (A -> word):
----------------------------------------
  NP -> 'astronomers'  [0.1]
  NP -> 'ears'  [0.18]
  NP -> 'saw'  [0.04]
  NP -> 'stars'  [0.18]
  NP -> 'telescopes'  [0.1]
  P -> 'with'  [1.0]
  V -> 'saw'  [1.0]

PARSING SENTENCE: astronomers saw stars with telescopes
INSIDE ALGORITHM

Sentence: astronomers saw stars with telescopes
Length: 5

The inside probability α(A, i, j) represents:
P(A =>* w_i w_{i+1} ... w_j)
i.e., the probability that non-terminal A generates words i through j

----------------------------------------------------------------------
STEP 1: Base Case (Lexical Rules) - Spans of Length 1
----------------------------------------------------------------------

Position 0, word = 'astronomers':
  α(NP, 0, 0) = P(NP -> 'astronomers') = 0.1

Pos